In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
df = pd.read_csv("../data/Retail_Transaction_Dataset.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (99460, 10)


,CustomerID,ProductID,Quantity,Price,TransactionDate,PaymentMethod,StoreLocation,ProductCategory,DiscountApplied(%),TotalAmount
0,109318,C,7,80.079844,12/26/2025 12:32,Cash,"176 Andrew Cliffs\nBaileyfort, HI 93354",Books,18.677100,455.862764
1,993229,C,4,75.195229,08/05/2025 00:00,Cash,"11635 William Well Suite 809\nEast Kara, MT 19483",Home Decor,14.121365,258.306546
2,579675,A,8,31.528816,03/11/2026 18:51,Cash,"910 Mendez Ville Suite 909\nPort Lauraland, MO...",Books,15.943701,212.015651
3,799826,D,5,98.880218,10/27/2025 22:00,PayPal,"87522 Sharon Corners Suite 500\nLake Tammy, MO...",Books,6.686337,461.343769
4,121413,A,7,93.188512,12/22/2025 11:38,Cash,"0070 Michelle Island Suite 143\nHoland, VA 80142",Electronics,4.030096,626.030484


In [3]:
df["TotalAmount"] = pd.to_numeric(
    df["TotalAmount"],
    errors="coerce"
)

df = df.dropna(subset=["TotalAmount"])

print("Valid Transactions:", len(df))

print("\nTotalAmount Statistics:")
print(df["TotalAmount"].describe())

Valid Transactions: 99460

TotalAmount Statistics:
count    99460.000000
mean       248.349743
std        184.599053
min          8.274825
25%         95.136619
50%        200.315564
75%        362.027687
max        896.141242
Name: TotalAmount, dtype: float64


In [4]:
# Milestone 2: Mean ± 3 Standard Deviations

mean_amount = df["TotalAmount"].mean()
std_amount = df["TotalAmount"].std()

upper_limit = mean_amount + (3 * std_amount)
lower_limit = mean_amount - (3 * std_amount)

baseline_mask = (
    (df["TotalAmount"] > upper_limit) |
    (df["TotalAmount"] < lower_limit)
)

baseline_count = baseline_mask.sum()
baseline_rate = (baseline_count / len(df)) * 100

print("=" * 60)
print("MILESTONE 2 BASELINE - 3 SIGMA")
print("=" * 60)

print(f"Mean              : {mean_amount:.2f}")
print(f"Standard Deviation: {std_amount:.2f}")
print(f"Lower Limit       : {lower_limit:.2f}")
print(f"Upper Limit       : {upper_limit:.2f}")
print(f"Anomalies         : {baseline_count}")
print(f"Anomaly Rate      : {baseline_rate:.3f}%")

MILESTONE 2 BASELINE - 3 SIGMA
Mean              : 248.35
Standard Deviation: 184.60
Lower Limit       : -305.45
Upper Limit       : 802.15
Anomalies         : 393
Anomaly Rate      : 0.395%


In [5]:
# Milestone 3: Refined IQR-Based Anomaly Detection

Q1 = df["TotalAmount"].quantile(0.25)
Q3 = df["TotalAmount"].quantile(0.75)

IQR = Q3 - Q1

iqr_multiplier = 1.75

refined_lower = Q1 - (iqr_multiplier * IQR)
refined_upper = Q3 + (iqr_multiplier * IQR)

print("=" * 60)
print("MILESTONE 3 - REFINED IQR MODEL")
print("=" * 60)

print(f"Q1             : {Q1:.2f}")
print(f"Q3             : {Q3:.2f}")
print(f"IQR            : {IQR:.2f}")
print(f"IQR Multiplier : {iqr_multiplier}")
print(f"Lower Limit    : {refined_lower:.2f}")
print(f"Upper Limit    : {refined_upper:.2f}")

MILESTONE 3 - REFINED IQR MODEL
Q1             : 95.14
Q3             : 362.03
IQR            : 266.89
IQR Multiplier : 1.75
Lower Limit    : -371.92
Upper Limit    : 829.09


In [6]:
df["Anomaly"] = np.where(
    (df["TotalAmount"] < refined_lower) |
    (df["TotalAmount"] > refined_upper),
    "Anomaly",
    "Normal"
)

refined_mask = df["Anomaly"] == "Anomaly"

refined_anomalies = df[refined_mask].copy()

refined_count = len(refined_anomalies)
refined_rate = (refined_count / len(df)) * 100

print("=" * 60)
print("IMPROVED ANOMALY DETECTION RESULTS")
print("=" * 60)

print(f"Total Transactions : {len(df)}")
print(f"Anomalies Found    : {refined_count}")
print(f"Anomaly Rate       : {refined_rate:.3f}%")

IMPROVED ANOMALY DETECTION RESULTS
Total Transactions : 99460
Anomalies Found    : 206
Anomaly Rate       : 0.207%


In [7]:
# Compare Milestone 2 and Milestone 3

both_flagged = (
    baseline_mask & refined_mask
).sum()

baseline_only = (
    baseline_mask & ~refined_mask
).sum()

refined_only = (
    ~baseline_mask & refined_mask
).sum()

retention = (both_flagged / baseline_count) * 100

alert_reduction = (
    (baseline_count - refined_count) / baseline_count
) * 100

print("=" * 65)
print("MILESTONE 2 vs MILESTONE 3 VALIDATION")
print("=" * 65)

print(f"M2 Anomalies          : {baseline_count}")
print(f"M3 Anomalies          : {refined_count}")
print(f"Both Methods Flagged  : {both_flagged}")
print(f"M2 Only               : {baseline_only}")
print(f"M3 Only               : {refined_only}")

print(f"\nBaseline Alerts Retained : {retention:.2f}%")
print(f"Alert Reduction          : {alert_reduction:.2f}%")

MILESTONE 2 vs MILESTONE 3 VALIDATION
M2 Anomalies          : 393
M3 Anomalies          : 206
Both Methods Flagged  : 206
M2 Only               : 187
M3 Only               : 0

Baseline Alerts Retained : 52.42%
Alert Reduction          : 47.58%


In [8]:
# Analyze the alerts removed by the improved detector

removed_alerts = df[
    baseline_mask & ~refined_mask
].copy()

print("=" * 65)
print("RETAINED VS REMOVED ALERTS")
print("=" * 65)

print("\nRetained Anomalies:")
print(
    refined_anomalies["TotalAmount"].describe()
)

print("\nRemoved M2 Alerts:")
print(
    removed_alerts["TotalAmount"].describe()
)

RETAINED VS REMOVED ALERTS

Retained Anomalies:
count    206.000000
mean     855.233111
std       18.314654
min      829.295550
25%      838.467468
50%      853.102205
75%      867.146487
max      896.141242
Name: TotalAmount, dtype: float64

Removed M2 Alerts:
count    187.000000
mean     814.384201
std        7.519889
min      802.241629
25%      808.569416
50%      813.525178
75%      820.088669
max      828.993645
Name: TotalAmount, dtype: float64


In [9]:
retained_mean = refined_anomalies["TotalAmount"].mean()
removed_mean = removed_alerts["TotalAmount"].mean()

mean_difference = retained_mean - removed_mean

mean_difference_percent = (
    mean_difference / removed_mean
) * 100

print("=" * 65)
print("FINAL ANOMALY DETECTION VALIDATION")
print("=" * 65)

print(f"M2 Anomalies              : {baseline_count}")
print(f"M3 Refined IQR Anomalies  : {refined_count}")

print(f"\nAlerts Reduced            : "
      f"{baseline_count - refined_count}")

print(f"Alert Reduction           : "
      f"{alert_reduction:.2f}%")

print(f"Baseline Alerts Retained  : "
      f"{retention:.2f}%")

print(f"M3-Only Alerts            : "
      f"{refined_only}")

print(f"\nMean Retained Amount      : "
      f"{retained_mean:.2f}")

print(f"Mean Removed Amount       : "
      f"{removed_mean:.2f}")

print(f"Mean Difference           : "
      f"{mean_difference:.2f}")

print(f"Difference Percentage     : "
      f"{mean_difference_percent:.2f}%")

FINAL ANOMALY DETECTION VALIDATION
M2 Anomalies              : 393
M3 Refined IQR Anomalies  : 206

Alerts Reduced            : 187
Alert Reduction           : 47.58%
Baseline Alerts Retained  : 52.42%
M3-Only Alerts            : 0

Mean Retained Amount      : 855.23
Mean Removed Amount       : 814.38
Mean Difference           : 40.85
Difference Percentage     : 5.02%


In [10]:
print("=" * 65)
print("FINAL IMPROVED ANOMALIES")
print("=" * 65)

print(
    refined_anomalies[
        [
            "CustomerID",
            "ProductID",
            "Quantity",
            "Price",
            "TransactionDate",
            "TotalAmount",
            "Anomaly"
        ]
    ].head(20)
)

FINAL IMPROVED ANOMALIES
       CustomerID ProductID  Quantity      Price   TransactionDate  \
460        493138         D         9  99.999284   9/27/2025 23:40   
1071       456529         A         9  98.316860   7/24/2025 12:54   
1081        70193         A         9  99.888441   3/28/2026 19:34   
1427       228232         B         9  94.661630  02/01/2026 18:41   
1832        24397         D         9  98.066472  10/02/2025 17:36   
3459       734916         C         9  99.216607  12/18/2025 13:18   
4653         9300         A         9  97.238897  03/11/2026 02:28   
5105       107740         D         9  97.840257  11/26/2025 21:22   
5396       449525         D         9  94.616104  08/04/2025 20:37   
6258       826828         A         9  98.842792    1/25/2026 4:39   
6474       107574         C         9  96.942985  03/01/2026 23:13   
6856       207212         C         9  98.839478  11/19/2025 17:15   
7535       731609         C         9  96.154621  01/01/2026 08:2

In [11]:
output_columns = [
    "CustomerID",
    "ProductID",
    "Quantity",
    "Price",
    "TransactionDate",
    "TotalAmount",
    "Anomaly"
]

final_output = df[output_columns].copy()

final_output.to_csv(
    "../data/improved_anomaly_detection.csv",
    index=False
)

print("Improved anomaly detection data saved successfully.")
print("../data/improved_anomaly_detection.csv")

Improved anomaly detection data saved successfully.
../data/improved_anomaly_detection.csv


In [12]:
iqr_model = {
    "method": "IQR",
    "multiplier": iqr_multiplier,
    "Q1": Q1,
    "Q3": Q3,
    "IQR": IQR,
    "lower_limit": refined_lower,
    "upper_limit": refined_upper
}

joblib.dump(
    iqr_model,
    "../models/improved_anomaly_detection.pkl"
)

print("=" * 65)
print("DAY 8 - ANOMALY DETECTION SUMMARY")
print("=" * 65)

print("\nBaseline Method:")
print("Mean ± 3 Standard Deviations")

print("\nImproved Method:")
print(f"IQR-based detection ({iqr_multiplier} × IQR)")

print("\nValidation:")
print(f"M2 Anomalies       : {baseline_count}")
print(f"M3 Anomalies       : {refined_count}")
print(f"Alert Reduction    : {alert_reduction:.2f}%")
print(f"Alerts Retained    : {retention:.2f}%")
print(f"M3-Only Alerts     : {refined_only}")

print("\nOutput Files:")
print("../data/improved_anomaly_detection.csv")
print("../models/improved_anomaly_detection.pkl")

DAY 8 - ANOMALY DETECTION SUMMARY

Baseline Method:
Mean ± 3 Standard Deviations

Improved Method:
IQR-based detection (1.75 × IQR)

Validation:
M2 Anomalies       : 393
M3 Anomalies       : 206
Alert Reduction    : 47.58%
Alerts Retained    : 52.42%
M3-Only Alerts     : 0

Output Files:
../data/improved_anomaly_detection.csv
../models/improved_anomaly_detection.pkl
